# Project 02 – StyleGAN2 Face Generation
Progressive training: **256 → 512 → 1024**

Fill in every `── USER CONFIG ──` block before running.

## 1. Environment setup

In [ ]:
!pip install -q wandb pytorch-fid pyyaml scikit-image onnx
!pip install -q 'torch>=2.2.0' torchvision --index-url https://download.pytorch.org/whl/cu121 --upgrade
import torch
print('CUDA:', torch.cuda.is_available(),
      ' device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A')
print('Torch:', torch.__version__)

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── USER CONFIG ──────────────────────────────────────────────────────────────
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/project02'  # checkpoint backup destination
# ────────────────────────────────────────────────────────────────────────────

import os
os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
print('Drive backup dir:', DRIVE_PROJECT_DIR)

## 3. Clone repository

In [ ]:
# ── USER CONFIG ──────────────────────────────────────────────────────────────
REPO_URL    = 'https://github.com/jyun-chae/project02.git'  # your GitHub repo
REPO_BRANCH = 'main'
# ────────────────────────────────────────────────────────────────────────────

import os, subprocess, sys

REPO_DIR = '/content/project02'

if os.path.exists(os.path.join(REPO_DIR, '.git')):
    print('Repo already cloned — pulling latest…')
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', REPO_BRANCH], check=True)
else:
    print('Cloning repo…')
    subprocess.run(
        ['git', 'clone', '--branch', REPO_BRANCH, '--depth', '1', REPO_URL, REPO_DIR],
        check=True,
    )

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Verify key modules are importable
from src.models.generator import StyleGAN2Generator
from src.models.discriminator import StyleGAN2Discriminator
from src.training.trainer import Trainer
from src.data.dataset import build_dataloader
from src.utils.fid_score import ValidFIDCache
from parallel_unzip import parallel_unzip
print('All modules imported OK.')

## 4. Extract FFHQ data

In [ ]:
# ── USER CONFIG ──────────────────────────────────────────────────────────────
TRAIN_ZIP = '/content/drive/MyDrive/project02/data/train.zip'
VALID_ZIP = '/content/drive/MyDrive/project02/data/valid.zip'
DATA_ROOT = '/content/ffhq'
# ────────────────────────────────────────────────────────────────────────────

from pathlib import Path

train_dir = Path(DATA_ROOT) / 'train'
valid_dir = Path(DATA_ROOT) / 'valid'

if not train_dir.exists() or len(list(train_dir.iterdir())) < 100:
    print('Extracting train zip…')
    parallel_unzip(TRAIN_ZIP, train_dir, strip_dirs=True, stage_local=True)

if not valid_dir.exists() or len(list(valid_dir.iterdir())) < 100:
    print('Extracting valid zip…')
    parallel_unzip(VALID_ZIP, valid_dir, strip_dirs=True, stage_local=True)

TRAIN_ROOT = str(train_dir)
VALID_ROOT = str(valid_dir)
print(f'Train: {len(list(train_dir.iterdir()))} images  |  Valid: {len(list(valid_dir.iterdir()))} images')

## 5. WandB login

In [ ]:
import wandb

# ── USER CONFIG ──────────────────────────────────────────────────────────────
WANDB_API_KEY = 'YOUR_WANDB_API_KEY_HERE'
WANDB_PROJECT = 'project02-stylegan2'
WANDB_ENTITY  = None  # your WandB username/org, or None for default
# ────────────────────────────────────────────────────────────────────────────

wandb.login(key=WANDB_API_KEY)
print('WandB logged in')

## 6. Config helper

In [ ]:
import yaml
from types import SimpleNamespace
import torch

def load_cfg(yaml_path: str) -> SimpleNamespace:
    with open(yaml_path) as f:
        return SimpleNamespace(**yaml.safe_load(f))

CFG_DIR  = f'{REPO_DIR}/configs'
CKPT_DIR = '/content/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
device = torch.device('cuda')

## 7. Training – Stage 1: 256×256
Train from scratch.

In [ ]:
cfg256 = load_cfg(f'{CFG_DIR}/train_256.yaml')

train_loader = build_dataloader(TRAIN_ROOT, 'train', cfg256.resolution, cfg256.batch_size, cfg256.num_workers)
valid_loader = build_dataloader(VALID_ROOT, 'valid', cfg256.resolution, cfg256.batch_size, cfg256.num_workers, aug=False)

run256 = wandb.init(
    project=WANDB_PROJECT, entity=WANDB_ENTITY,
    name='train-256', config=vars(cfg256), resume='allow',
)

trainer256 = Trainer(cfg256)
# trainer256.load(f'{CKPT_DIR}/ckpt_256_XXXXXXX.pth')  # ← uncomment to resume

trainer256.fit(
    train_loader, valid_loader,
    wandb_run=run256,
    drive_backup_dir=f'{DRIVE_PROJECT_DIR}/checkpoints',
    ckpt_dir=CKPT_DIR,
)
run256.finish()
CKPT_256 = f'{CKPT_DIR}/ckpt_256_final.pth'
print('Stage 1 done →', CKPT_256)

## 8. Training – Stage 2: 512×512
Fine-tune from 256 checkpoint.

In [ ]:
cfg512 = load_cfg(f'{CFG_DIR}/train_512.yaml')

train_loader_512 = build_dataloader(TRAIN_ROOT, 'train', cfg512.resolution, cfg512.batch_size, cfg512.num_workers)
valid_loader_512 = build_dataloader(VALID_ROOT, 'valid', cfg512.resolution, cfg512.batch_size, cfg512.num_workers, aug=False)

run512 = wandb.init(
    project=WANDB_PROJECT, entity=WANDB_ENTITY,
    name='train-512', config=vars(cfg512), resume='allow',
)

trainer512 = Trainer(cfg512)

# ── USER CONFIG: path to 256 final checkpoint ─────────────────────────────
CKPT_256 = f'{CKPT_DIR}/ckpt_256_final.pth'
# ────────────────────────────────────────────────────────────────────────────

state = torch.load(CKPT_256, map_location=device)
trainer512.G.load_from_lower_resolution(state['G'])
trainer512.D.load_state_dict(state['D'], strict=False)

trainer512.fit(
    train_loader_512, valid_loader_512,
    wandb_run=run512,
    drive_backup_dir=f'{DRIVE_PROJECT_DIR}/checkpoints',
    ckpt_dir=CKPT_DIR,
)
run512.finish()
CKPT_512 = f'{CKPT_DIR}/ckpt_512_final.pth'
print('Stage 2 done →', CKPT_512)

## 9. Training – Stage 3: 1024×1024
Fine-tune from 512 checkpoint.

In [ ]:
cfg1024 = load_cfg(f'{CFG_DIR}/train_1024.yaml')

train_loader_1024 = build_dataloader(TRAIN_ROOT, 'train', cfg1024.resolution, cfg1024.batch_size, cfg1024.num_workers)
valid_loader_1024 = build_dataloader(VALID_ROOT, 'valid', cfg1024.resolution, cfg1024.batch_size, cfg1024.num_workers, aug=False)

run1024 = wandb.init(
    project=WANDB_PROJECT, entity=WANDB_ENTITY,
    name='train-1024', config=vars(cfg1024), resume='allow',
)

trainer1024 = Trainer(cfg1024)

# ── USER CONFIG: path to 512 final checkpoint ─────────────────────────────
CKPT_512 = f'{CKPT_DIR}/ckpt_512_final.pth'
# ────────────────────────────────────────────────────────────────────────────

state = torch.load(CKPT_512, map_location=device)
trainer1024.G.load_from_lower_resolution(state['G'])
trainer1024.D.load_state_dict(state['D'], strict=False)

trainer1024.fit(
    train_loader_1024, valid_loader_1024,
    wandb_run=run1024,
    drive_backup_dir=f'{DRIVE_PROJECT_DIR}/checkpoints',
    ckpt_dir=CKPT_DIR,
)
run1024.finish()
CKPT_1024 = f'{CKPT_DIR}/ckpt_1024_final.pth'
print('Stage 3 done →', CKPT_1024)

## 10. Evaluate: FID on valid set

In [ ]:
# ── USER CONFIG ──────────────────────────────────────────────────────────────
EVAL_CKPT = CKPT_1024
EVAL_RES  = 1024
# ────────────────────────────────────────────────────────────────────────────

cfg_eval = load_cfg(f'{CFG_DIR}/train_{EVAL_RES}.yaml')
G_eval = StyleGAN2Generator(
    resolution=cfg_eval.resolution,
    z_dim=cfg_eval.z_dim,
    w_dim=cfg_eval.w_dim,
    channel_base=cfg_eval.channel_base,
    channel_max=cfg_eval.channel_max,
    mapping_layers=cfg_eval.mapping_layers,
).to(device)

state = torch.load(EVAL_CKPT, map_location=device)
G_eval.load_state_dict(state['G'])
G_eval.eval()
print(f'Generator parameters: {G_eval.count_parameters():,}')

valid_loader_eval = build_dataloader(VALID_ROOT, 'valid', EVAL_RES, 8, 4, aug=False)
fid_cache = ValidFIDCache(valid_loader_eval, device)
fid = fid_cache.compute(G_eval, n_gen=10000, batch_size=16)
print(f'FID (valid, 10k): {fid:.2f}')

## 11. Generate samples

In [ ]:
import matplotlib.pyplot as plt
from torchvision.utils import make_grid

mean_w = G_eval.mapping.mean_latent(n_samples=4096, device=str(device))

with torch.no_grad():
    z = torch.randn(16, cfg_eval.z_dim, device=device)
    imgs = G_eval(z, noise_mode='const', truncation=0.7, mean_w=mean_w)
    imgs = (imgs.clamp(-1, 1) + 1) / 2

grid = make_grid(imgs.cpu(), nrow=4, padding=2)
plt.figure(figsize=(14, 14))
plt.imshow(grid.permute(1, 2, 0).numpy())
plt.axis('off')
plt.title(f'StyleGAN2 {EVAL_RES}×{EVAL_RES}  truncation=0.7')
plt.tight_layout()
plt.savefig('/content/samples.png', dpi=150)
plt.show()

## 12. Export ONNX (for submission)

In [ ]:
ONNX_PATH = f'/content/generator_{EVAL_RES}.onnx'
onnx_params = G_eval.export_onnx(ONNX_PATH, batch_size=1)
print(f'ONNX parameters: {onnx_params:,} ({onnx_params/1e6:.3f}M)  [limit: 40M]')

import shutil
shutil.copy(ONNX_PATH, f'{DRIVE_PROJECT_DIR}/generator_{EVAL_RES}.onnx')
print('ONNX backed up to Drive.')

## 13. Parameter count sanity check

In [ ]:
print(f'{"res":>6}  {"G (M)":>8}  {"D (M)":>8}  {"G < 40M":>8}')
print('-' * 40)
for res in [256, 512, 1024]:
    G = StyleGAN2Generator(resolution=res, channel_base=65536, channel_max=512,
                           w_dim=640, mapping_layers=12)
    D = StyleGAN2Discriminator(resolution=res, channel_base=65536, channel_max=512)
    gp = G.count_parameters()
    dp = sum(p.numel() for p in D.parameters())
    ok = '✓' if gp < 40_000_000 else '✗ OVER'
    print(f'{res:>6}  {gp/1e6:>8.3f}  {dp/1e6:>8.3f}  {ok:>8}')